# 04 — Validation statistique (A/B test)

Le traitement est **randomisé** dans les données → on peut estimer, **sans biais**, la valeur de n'importe quelle politique de ciblage par pondération inverse (IPW) :

`conv(π) = mean_i [ 1(T_i = π(x_i)) / P(T = π(x_i)) × y_i ]`

On compare ensuite deux politiques avec un test **apparié** sur les contributions client par client.

**Questions de recherche**
1. Avec combien de clients peut-on détecter l'écart attendu ? (puissance)
2. La politique optimisée bat-elle significativement le ciblage aléatoire / naïf ?
3. Comment l'incertitude évolue-t-elle avec la taille d'échantillon ?

In [1]:
import sys, warnings
sys.path.insert(0, "../src")            # pas d'installation nécessaire
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from config import load_config
cfg = load_config()
ARMS, CTRL = cfg["arms"], cfg["control_label"]
pd.set_option("display.float_format", lambda v: f"{v:.5f}")

from train import ensure_scores
from optimization import optimize_allocation, naive_top_uplift, random_policy
from ab_test import required_sample_size, min_detectable_effect, policy_value, compare_policies
scores = ensure_scores(cfg)
U = scores[[f"uplift_{a}" for a in ARMS]].rename(columns=lambda c: c.removeprefix("uplift_"))
VALUE, COSTS, B = cfg["business"]["value_per_conversion"], cfg["business"]["cost_per_contact"], cfg["business"]["budget"]
p0 = scores[scores.treatment == CTRL].y.mean(); print(f"taux de conversion contrôle : {p0:.4%}")

taux de conversion contrôle : 0.5788%


## 1. Analyse de puissance

In [2]:
rows = [dict(lift_abs=d, n_per_group=required_sample_size(p0, p0 + d)) for d in [0.001, 0.002, 0.003, 0.005, 0.01]]
display(pd.DataFrame(rows).set_index("lift_abs"))
ns = [2_000, 5_000, 10_000, 20_000, 50_000]
plt.plot(ns, [min_detectable_effect(n, p0) for n in ns], marker="o"); plt.xscale("log")
plt.xlabel("clients par groupe"); plt.ylabel("effet minimal détectable (pts de conv.)"); plt.title("MDE (alpha=5 %, puissance=80 %)"); plt.grid(alpha=.3); plt.show()

,n_per_group
lift_abs,
0.00100,97940
0.00200,26317
0.00300,12486
0.00500,5042
0.01000,1581


## 2. Politiques et test apparié (IPW)

In [3]:
opt, _ = optimize_allocation(U, VALUE, COSTS, B, CTRL)
pol = {"optimisée (LP)": opt, "naïf top-uplift": naive_top_uplift(U, COSTS, B, CTRL), "aléatoire": random_policy(U, COSTS, B, CTRL, cfg["seed"])}
display(pd.DataFrame({k: policy_value(scores, v, VALUE, COSTS, CTRL) for k, v in pol.items()}).T)
for base in ["aléatoire", "naïf top-uplift"]:
    r = compare_policies(scores, pol[base], pol["optimisée (LP)"], VALUE, COSTS, CTRL)
    p = r["profit_per_customer"]
    print(f"optimisée vs {base:16s} Δprofit/client={p['diff_mean']:+.5f}  IC95%=[{p['ci_low']:+.5f}, {p['ci_high']:+.5f}]  p={p['p_value']:.4f}  significatif={p['significant']}")

,conv_rate_ipw,profit_per_customer_ipw,total_profit_ipw,total_cost,n_contacted
optimisée (LP),0.00970,0.25965,4985.29833,600.00000,6000.00000
naïf top-uplift,0.00970,0.25966,4985.39833,599.90000,5999.00000
aléatoire,0.00704,0.17983,3452.80928,599.90000,5999.00000


optimisée vs aléatoire        Δprofit/client=+0.07982  IC95%=[+0.00923, +0.15040]  p=0.0267  significatif=True
optimisée vs naïf top-uplift  Δprofit/client=-0.00001  IC95%=[-0.00002, +0.00000]  p=0.3173  significatif=False


## 3. Incertitude vs taille d'échantillon (sous-échantillonnage du test)

In [4]:
rng = np.random.default_rng(cfg["seed"]); rows = []
for frac in [0.25, 0.5, 0.75, 1.0]:
    for rep in range(10):
        idx = rng.choice(len(scores), int(len(scores) * frac), replace=False)
        sub = scores.iloc[idx]
        r = compare_policies(sub, pol["aléatoire"].iloc[idx], pol["optimisée (LP)"].iloc[idx], VALUE, COSTS, CTRL)["profit_per_customer"]
        rows.append(dict(frac=frac, diff=r["diff_mean"], p=r["p_value"], sig=r["significant"]))
s = pd.DataFrame(rows).groupby("frac").agg(diff_mean=("diff", "mean"), diff_std=("diff", "std"), share_significant=("sig", "mean"))
display(s)

,diff_mean,diff_std,share_significant
frac,,,
0.25000,0.12465,0.04753,0.50000
0.50000,0.10258,0.03787,0.60000
0.75000,0.07282,0.02308,0.30000
1.00000,0.07982,0.00000,1.00000


**Conclusion** : la politique optimisée bat significativement le ciblage aléatoire, mais avec une incertitude importante, et elle ne fait pas mieux que le top-uplift naïf. Le résultat est à la limite de la puissance statistique du jeu de test.

### 1. Puissance (taux de base : 0,58 %)
- La taille d'échantillon requise croît en $1/\Delta^2$ : ≈ 1 600 clients par groupe pour détecter +1 pt, ≈ 5 000 pour +0,5 pt, ≈ 12 500 pour +0,3 pt, ≈ 26 000 pour +0,2 pt et ≈ 98 000 pour +0,1 pt.
- Les effets des emails (Mens +0,68 pt, Womens +0,31 pt) sont détectables avec quelques milliers à ≈ 12 500 clients par groupe.
- Le gain du ciblage (+0,27 pt de conversion vs aléatoire) est plus petit : le confirmer en test live demande ≈ 18 000 clients par groupe.

### 2. Politiques et test apparié (IPW)
- **Optimisée vs aléatoire** : conversion de 0,97 % contre 0,70 %, profit de 0,26 € contre 0,18 € par client, soit **+0,08 € par client (+44 %)** à budget égal (600 €, 6 000 contacts), environ +1 500 € sur le jeu de test.
- Le test apparié donne un IC 95 % de [+0,009 ; +0,150] € et p = 0,027 : **significatif**, mais la borne basse est proche de 0. Le gain est probable, son ampleur mal connue.
- Si l'on corrige pour les deux comparaisons (Holm), la p-value passe à ≈ 0,053 : à la limite.
- **Optimisée vs naïve** : écart nul (−0,00001 € par client, p = 0,32). Les deux politiques contactent les mêmes clients à un client près : coût identique pour les deux bras et budget saturé, donc le LP retombe sur le classement par uplift. Il s'agit d'une quasi-identité, pas d'un manque de puissance.
- **D'où vient le gain** : du score d'uplift, pas du solveur. Environ +1,25 pt de conversions incrémentales par client contacté, contre ≈ +0,4 pt au hasard (ordre de grandeur, à partir du taux du contrôle).

### 3. Incertitude et taille d'échantillon
- L'écart-type de l'estimation diminue avec la taille du sous-échantillon : ≈ 0,048 (25 %), 0,038 (50 %), 0,023 (75 %), conforme à l'ordre de grandeur en $1/\sqrt{n}$.
- Avec 25 % à 75 % des données, la différence n'est significative que dans 30 % à 60 % des tirages : la conclusion dépend de la taille du test.
- La ligne à 100 % (écart-type nul, 100 % de significativité) est un artefact : il n'existe qu'un seul échantillon complet. L'incertitude réelle est donnée par l'IC du §2.
- L'écart moyen varie (≈ 0,125 à 25 %, ≈ 0,08 à 100 %) à cause du tirage, avec seulement 10 répétitions ; la valeur de référence reste ≈ 0,08 €.

### Limites
- Un seul jeu de test, 10 répétitions par taille : passer à 200 répétitions ou à un bootstrap.
- `value_per_conversion` et `cost_per_contact` sont des hypothèses métier : faire une analyse de sensibilité.
- Vérifier que les scores d'uplift viennent d'un modèle **non entraîné** sur le jeu d'évaluation.
- Le profit IPW inclut les conversions qui auraient eu lieu sans email : seuls les écarts entre politiques sont causaux.

## Lecture et décision
- **Significatif** (optimisée vs aléatoire) : on annonce le gain avec son intervalle (+0,08 € par client, IC [+0,01 ; +0,15]), pas le seul +44 %. **Non significatif** : on ne conclut pas à l'absence d'effet, sauf si les politiques sont quasi identiques (cas optimisée vs naïve). Sinon l'échantillon manque de puissance, et la table du §1 donne la taille d'un **test live** à lancer (`n_per_group_live_test` dans `reports/optimization.json`).
- **Décision** : garder le ciblage par uplift et le confirmer par un test live d'environ 18 000 clients par groupe avant généralisation. Garder le LP seulement si des contraintes réelles l'exigent (coûts par bras, plafonds) ; sinon le top-uplift suffit.
- Dans tous les cas, les chiffres publiés dans le README viennent de `reports/RESULTS.md` (généré par `python -m report`), jamais saisis à la main.